## Környezet ellenőrzése

In [ ]:
!conda info --envs

## Notebook tartalomjegyzéke

Felhasznált adathalmaz: **Online Retail II UCI**
https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data  


0\. Adatbetöltés és parquet konverzió

1\. Adattisztítás és alap EDA SQLite segítségével

2\. RFM Feature Engineering: alapmutatók kiszámítása.

3\. Outlier-kezelés és Skálázás: (Pl. StandardScaler vagy MinMaxScaler a K-means előtt, mert a távolság alapú algoritmusok érzékenyek a léptékre).

4\. K-means klaszterezés: Szegmentálás és a szegmensek profilozása.

5\. Kiterjesztett EDA: A klaszterek vizualizálása (pl. Snake plot vagy Heatmap).

6\. XGBoost Célváltozó tervezés: Címkézés (pl. Churn: 0 vagy 1) egy adott időpont alapján.

7\. Modellezés és SHAP: Tanítás, tesztelés és a modell magyarázhatósága (melyik RFM faktor a legfontosabb a churnnél?).

8\. Üzleti kiértékelés: Mit kezdjünk az egyes csoportokkal? (Pl. "Hűségeseknek kedvezmény", "Lemorzsolódóknak re-engagement kampány").

9\. Export: Streamlit-hez vagy egyéb BI eszközhöz.

## Műszaki és verziókezelési megjegyzések

**Adatformátum (Parquet):** A projekt során a nyers CSV adatokat az első lépésben Apache Parquet formátumba transzformálom és a data/processed/ mappában tárolom az alábbi, data engineeringben ismert előnyök miatt:


Hatékonyság: Az oszlop-alapú tárolás jelentősen gyorsabb beolvasást és kisebb memóriahasználatot tesz lehetővé a feature engineering során.


Típusbiztonság: A Parquet megőrzi a sémát (pl. InvoiceDate datetime vagy Customer ID integer típusát), így elkerülhetők a CSV-re jellemző ismételt típuskonverziós hibák.

**Verziókezelés (nbstripout):** A notebook tisztán tartása érdekében ``nbstripout`` használatával dolgozom. Ez a git-filter biztosítja, hogy a távoli repóba (GitHub) kizárólag a forráskód kerüljön be, a futtatási metaadatok és nagyméretű vizuális kimenetek nélkül. Ez elősegíti a tiszta code review folyamatot és megakadályozza a bináris szemét felhalmozódását a verziótörténetben.

---
## 0. Adatbetöltés és Parquet-konverzió

A nyers adathalmaz betöltése során elsődlegesen az automatizált megoldásra törekszünk. Mivel azonban a specifikus UCI API korlátokba ütközött, a kód egy robusztus ellenőrző logikát használ: ha a nyers adatok hiányoznak, pontos instrukciókat ad a manuális beszerzéshez, majd elvégzi a Parquet konverziót az optimális további feldolgozáshoz.

A `0.2` cella idempotens: ha a tisztított Parquet fájl már létezik, automatikusan kihagyja az ismételt letöltést és konverziót.

In [ ]:
# ============================================================
# 0.1 – Könyvtárak, konstansok és adat-ellenőrzés
# ============================================================
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Útvonalak beállítása
PROJECT_ROOT = Path(".").resolve()
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Mappastruktúra létrehozása
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Fájlnevek
RAW_FILE = RAW_DIR / "online_retail_II.csv"
PARQUET_OUT = PROCESSED_DIR / "online_retail_raw.parquet"

print(f"PROJECT_ROOT:  {PROJECT_ROOT}")
print(f"RAW_FILE:      {RAW_FILE}")
print(f"PARQUET_OUT:   {PARQUET_OUT}\n")

# --- Klónozás utáni állapot ellenőrzése ---
if not PARQUET_OUT.exists() and not RAW_FILE.exists():
    error_msg = (
        "\n" + "="*80 + "\n"
        "HIÁNYZÓ ADATHALMAZ!\n"
        "A hivatalos UCI API nem támogatja ezt a specifikus adathalmazt, így manuális letöltés szükséges.\n\n"
        "Kérlek, kövesd az alábbi lépéseket a folytatáshoz:\n"
        "1. Töltsd le a zip fájlt a Kaggle-ről:\n"
        "   https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data\n"
        "2. Csomagold ki, és az 'online_retail_II.csv' fájlt helyezd el ide:\n"
        f"   {RAW_FILE}\n"
        "3. Futtasd újra ezt a cellát!\n"
        + "="*80 + "\n"
    )
    print(error_msg)
    raise FileNotFoundError("A nyers adathalmaz nem található. Kövesd a fenti utasításokat!")
else:
    print("Fájlrendszer ellenőrizve: a szükséges adatfájlok rendelkezésre állnak!")

In [ ]:
# ============================================================
# 0.2 – Nyers CSV betöltése és Parquet-be konvertálása
# ============================================================

if PARQUET_OUT.exists():
    print(f"Parquet már létezik, kihagyjuk a konverziót: {PARQUET_OUT}")
else:
    dtype_map = {
        "Invoice":     "string",
        "StockCode":   "string",
        "Description": "string",
        "Quantity":    "float64",
        "Price":       "float64",
        "Customer ID": "float64",
        "Country":     "string",
    }
    parse_dates = ["InvoiceDate"]

    print(f"CSV fájl betöltése innen: {RAW_FILE} ... (ez eltarthat egy percig)")
    df_raw = pd.read_csv(
        RAW_FILE,
        dtype=dtype_map,
        parse_dates=parse_dates,
        encoding="utf-8",
    )
    
    print(f"Összesített sorok (nyers): {len(df_raw):,}")

    # Customer ID: float -> nullable Int64 (megőrzi a NaN-okat is)
    df_raw["Customer ID"] = df_raw["Customer ID"].astype("Int64")

    # Parquet mentés
    df_raw.to_parquet(PARQUET_OUT, compression="snappy", index=False)

    size_mb = PARQUET_OUT.stat().st_size / 1_048_576
    print(f"\nParquet mentve: {PARQUET_OUT}")
    print(f"Fájlméret:      {size_mb:.1f} MB")
    print(f"Sorok:          {len(df_raw):,} | Oszlopok: {df_raw.shape[1]}")
    print(f"\nSéma:\n{df_raw.dtypes}")

## 1. Adattisztítás

A Customer Lifetime Value (CLV) modellezés és az RFM szegmentáció alapfeltétele, hogy az adatokat vásárlói szinten tudjuk aggregálni. Ennek megfelelően az alábbi tisztítási lépéseket végezzük el:

- **Anonim tranzakciók eltávolítása:** A hiányzó `Customer ID`-val rendelkező sorok kötelezően eldobandók.
- **Érvénytelen értékek szűrése:** A 0 vagy negatív `Price` értékkel rendelkező sorok eltávolítása.
- **Duplikátumok szűrése:** Az azonos tartalmú, ismétlődő sorok törlése.
- **Adminisztratív és nem termék kódok eltávolítása:** A nyers adatokon végzett előzetes **SQLite-os adatfeltárás** során kiderült, hogy az adathalmazban jelentős számú, torzító hatású adminisztratív bejegyzés található (pl. `POST`, `M`, `BANK CHARGES`, `AMAZONFEE`). Ezek mellé soroljuk a `C2` (Carriage) kódot is, amely egy kiugróan magas árú, fix tengerentúli fuvardíjra utal. Mivel ezek nem kézzelfogható termékek (és gyakran extrém negatív korrekciós értékeket képviselnek), teljesen kiszűrjük őket, hogy ne torzítsák a termékalapú kosárelemzést és a CLV-t.
- **Fejlesztői teszt bejegyzések szűrése:** Szintén az SQLite-os lekérdezések buktatták le a rendszerben maradt teszt tranzakciókat (pl. `TEST001`, `TEST002` cikkszámok, vagy `test` leírások). Mivel ezek irreális árakkal és mennyiségekkel rontanák a statisztikákat, a `StockCode` és a `Description` oszlopok együttes vizsgálatával az összeset eldobjuk.

*Megjegyzés: A sztornó/visszaküldött tételeket ('C' prefix az Invoice oszlopban) ebben a fázisban szándékosan megtartjuk, mivel ezek a későbbiekben negatív feature-ként szolgálnak a vásárlói érték (CLV) számításánál.*

In [ ]:
# ============================================================
# 1. Adattisztítás
# ============================================================
import pandas as pd

print("Adattisztítás megkezdése...\n")

# 1. Nyers Parquet betöltése
df = pd.read_parquet(PARQUET_OUT)
eredeti_sorok_szama = len(df)
print(f"Kiinduló sorok száma: {eredeti_sorok_szama:,}")
print("-" * 50)

# 2. Hiányzó Customer ID-k eldobása
df = df.dropna(subset=["Customer ID"])
id_nelkuli_sorok = eredeti_sorok_szama - len(df)
print(f"✔️ Nincsenek anonim vásárlók. (Eldobva: {id_nelkuli_sorok:,} sor)")

# Opcionális, de ajánlott: Customer ID stringgé alakítása
df["Customer ID"] = df["Customer ID"].astype(str)
print("✔️ A Customer ID biztonságos string formátumban van.")

# 3. Érvénytelen árak szűrése (Price <= 0)
ervenytelen_ar_maszk = df["Price"] <= 0
ervenytelen_ar_sorok = ervenytelen_ar_maszk.sum()
df = df[~ervenytelen_ar_maszk]
print(f"✔️ Az érvénytelen (0 vagy negatív) árak eltűntek. (Eldobva: {ervenytelen_ar_sorok:,} sor)")

# --- Sztornók megtartása ---
print("✔️ A sztornók okosan megmaradnak a feature engineeringhez (Quantity <= 0 szándékosan nincs szűrve).")

# 4. Nem termék jellegű tranzakciók kiszűrése
nem_termek_maszk = (
    df["StockCode"].astype(str).str.replace(" ", "").str.isalpha() | 
    (df["StockCode"].astype(str).str.upper() == "C2")
)
nem_termek_sorok = nem_termek_maszk.sum()
df = df[~nem_termek_maszk]
print(f"✔️ A 'BANK CHARGES' jellegű és 'C2' fuvardíj kódok célzottan eldobásra kerültek. (Eldobva: {nem_termek_sorok:,} sor)")

# 5. Teszt bejegyzések eltávolítása
teszt_maszk = (
    df["Description"].astype(str).str.contains("TEST", case=False, na=False) |
    df["StockCode"].astype(str).str.contains("TEST", case=False, na=False)
)
teszt_sorok = teszt_maszk.sum()
df = df[~teszt_maszk]
print(f"✔️ A rejtett, üres leírású tesztkódok is fennakadtak a szűrőn. (Eldobva: {teszt_sorok:,} sor)")

# 6. Duplikátumok eldobása
duplikatum_maszk = df.duplicated()
duplikatum_sorok = duplikatum_maszk.sum()
df = df[~duplikatum_maszk]
print(f"✔️ A duplikátumok eltávolítva. (Eldobva: {duplikatum_sorok:,} sor)")

# 7. Tisztított adatok mentése checkpointként
CLEANED_PARQUET = PROCESSED_DIR / "online_retail_cleaned.parquet"
df.to_parquet(CLEANED_PARQUET, compression="snappy", index=False)

vegleges_sorok_szama = len(df)
print("-" * 50)
print(f"Adattisztítás eredménye:")
print(f"Megmaradt tiszta sorok száma: {vegleges_sorok_szama:,} "
      f"(az eredeti {vegleges_sorok_szama/eredeti_sorok_szama*100:.1f}%-a)")
print(f"Tisztított adatok mentve: {CLEANED_PARQUET}")

In [ ]:
df.head()